# 1. generate monthly soil moisture anormalies

In [ ]:
import xarray as xr

# Load soil moisture datasets from different sources
somo = xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_somo.nc').layer1
gleam = xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_gleam.nc').SMs
esa = xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_eascci.nc").sm
era5 = xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/swvl1.nc").swvl1

# Define function to remove linear trend from data along specified dimension
def detrend_dim(da, dim, deg=1):
    # Calculate polynomial fit coefficients
    p = da.polyfit(dim=dim, deg=deg)
    # Generate fitted values based on polynomial coefficients
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    # Return detrended data by subtracting the fitted trend
    return da - fit

# Create lists for dataset names and corresponding data arrays
varlist_name = ["somo", "gleam", "esa", "era5"]
varlist = [somo, gleam, esa, era5]

# Process each dataset in the list
for i, var in enumerate(varlist):
    # Remove linear trend from the time series
    var_dt = detrend_dim(var, dim='time', deg=1)
    
    # Calculate monthly climatology (seasonal cycle)
    var_mean = var_dt.groupby('time.month').mean(dim='time')
    
    # Remove seasonal cycle to get anomalies
    var_anom = var_dt.groupby('time.month') - var_mean
    
    # Clean coordinate data by filtering invalid values
    var_anom['longitude'] = var_anom['longitude'].where(var_anom['longitude'] != -9999)
    var_anom['latitude'] = var_anom['latitude'].where(var_anom['latitude'] != -9999)
    
    # Save processed anomalies to NetCDF files
    var_anom.to_netcdf('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/' + varlist_name[i] + '.nc')
    
    # Print progress indicator
    print(i)

# 2. process the four soil moisture datasets 

In [1]:
import xarray as xr
somo=xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_somo.nc')
gleam=xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_gleam.nc')
esa=xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_eascci.nc")
era5=xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/swvl1.nc")

In [22]:
# Select a specific latitude range from the gleam dataset
# The slice operation selects latitudes from 78.75°N to 80.5°S (inclusive)
# Note: The negative value indicates southern hemisphere
gleam = gleam.sel(latitude=slice(78.75, -80.5))

(107, 240, 216)

In [23]:
# Import required libraries
from get_lat_lon import get_lat_lon  # Custom function to get latitude/longitude
import numpy as np
import xarray as xr

# Load pre-defined data arrays
considered_cells = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/considered_cells.npy")
sa_x_cell = np.load("/User/homes/fhuang/lac_cnn/sa_x_cell.npy")

# Get number of samples from the loaded data
nsample = sa_x_cell.shape[0]

# Initialize output array with NaN values for storing multi-feature data
idata = np.full((nsample, 216, 8), np.nan)

# Initialize lists to track cells with missing data
x_cell_nan = []
x_cell_nan_idx = []

# Counter variable (unused in current code)
k = 0

# Process each sample in the dataset
for i in range(nsample):
    # Get current cell index
    x_cell = sa_x_cell[i]
    
    # Extract latitude and longitude from considered_cells array
    lati, loni = considered_cells[:, x_cell]
    loni = loni - 120  # Adjust longitude index
    
    # Get actual latitude and longitude coordinates using custom function
    lat, lon = get_lat_lon(x_cell)
    
    # Adjust latitude index (1-based to 0-based indexing)
    lati = lati - 1
    
    # Store geographic information in output array
    idata[i, :, 0] = lati    # Latitude index
    idata[i, :, 1] = loni    # Longitude index
    idata[i, :, 2] = lat     # Actual latitude value
    idata[i, :, 3] = lon     # Actual longitude value
    
    # Extract and store soil moisture data from different datasets
    idata[i, :, 4] = somo.layer1.data[lati, loni, :]    # SOMo dataset layer1
    idata[i, :, 5] = gleam.SMs.data[lati, loni, :]      # GLEAM dataset SMs
    idata[i, :, 6] = esa.sm.data[:, lati, loni]         # ESA dataset soil moisture
    idata[i, :, 7] = era5.swvl1.data[:, lati, loni]     # ERA5 dataset soil water volume layer1
    
    # Print progress information for debugging/monitoring
    print(x_cell, i, lati, loni, lat, lon)

# Save the compiled multi-dataset soil moisture data to numpy file
np.save('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_4dataset.npy', idata)

5920 0 43 67 13.5 -79.5
5921 1 43 79 13.5 -61.5
5922 2 43 80 13.5 -60.0
6004 3 44 71 12.0 -73.5
6005 4 44 72 12.0 -72.0
6006 5 44 73 12.0 -70.5
6007 6 44 74 12.0 -69.0
6008 7 44 75 12.0 -67.5
6009 8 44 76 12.0 -66.0
6010 9 44 77 12.0 -64.5
6011 10 44 78 12.0 -63.0
6012 11 44 79 12.0 -61.5
6013 12 44 80 12.0 -60.0
6094 13 45 69 10.5 -76.5
6095 14 45 70 10.5 -75.0
6096 15 45 71 10.5 -73.5
6097 16 45 72 10.5 -72.0
6098 17 45 73 10.5 -70.5
6099 18 45 74 10.5 -69.0
6100 19 45 75 10.5 -67.5
6101 20 45 76 10.5 -66.0
6102 21 45 77 10.5 -64.5
6103 22 45 78 10.5 -63.0
6104 23 45 79 10.5 -61.5
6105 24 45 80 10.5 -60.0
6187 25 46 67 9.0 -79.5
6188 26 46 68 9.0 -78.0
6189 27 46 69 9.0 -76.5
6190 28 46 70 9.0 -75.0
6191 29 46 71 9.0 -73.5
6192 30 46 72 9.0 -72.0
6193 31 46 73 9.0 -70.5
6194 32 46 74 9.0 -69.0
6195 33 46 75 9.0 -67.5
6196 34 46 76 9.0 -66.0
6197 35 46 77 9.0 -64.5
6198 36 46 78 9.0 -63.0
6199 37 46 79 9.0 -61.5
6200 38 46 80 9.0 -60.0
6282 39 47 67 7.5 -79.5
6283 40 47 68 7.5 -78.0
6

In [48]:
# Load indices of samples containing NaN values from a pre-saved file
nan_idx = np.load('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/conv3d_res/nan_idx.npy')

# Commented out: Option to remove NaN samples from the main data array (idata)
# idata = np.delete(idata, nan_idx, axis=0)

# Load the original sample cell indices array
sa_x_cell = np.load("/User/homes/fhuang/lac_cnn/sa_x_cell.npy")

# Remove samples with NaN values from the cell indices array
# This ensures alignment between cleaned data and cell indices
sa_x_cell = np.delete(sa_x_cell, nan_idx, axis=0)

# Print the shapes of both arrays to verify dimensions match after cleaning
idata.shape, sa_x_cell.shape

((737, 216, 8), (737,))

# 3. fill esa cci gaps with era5-land soil moisture 

In [1]:
from get_lat_lon import get_lat_lon
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import xarray as xr
import numpy as np
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

def surrounding_pixel_mean(somo,esa,lat,lon):
    '''use surronding pixels' average value as the conter pixel's value'''
    esa_i_new_np=np.full((8,216),np.nan)
    for i in range(8):
        if i ==0:
            lat_s=lat-1.5
            lon_s=lon
        elif i==1:
            lat_s=lat+1.5
            lon_s=lon
        elif i==2:
            lat_s=lat
            lon_s=lon+1.5
        elif i==3:
            lat_s=lat
            lon_s=lon-1.5
        elif i==4:
            lat_s=lat-1.5
            lon_s=lon-1.5
        elif i==5:
            lat_s=lat+1.5
            lon_s=lon-1.5
        elif i==6:
            lat_s=lat+1.5
            lon_s=lon+1.5
        elif i==7:
            lat_s=lat-1.5
            lon_s=lon+1.5
        
        somo_i=somo.layer1.sel(latitude=lat_s,longitude=lon_s,method='nearest')
        esa_i=esa.sm.sel(latitude=lat_s,longitude=lon_s,method='nearest')
        std_esa=np.nanstd(esa_i)
        std_somo=np.nanstd(somo_i)
        mean_esa=np.nanmean(esa_i)
        mean_somo=np.nanmean(somo_i)
        esa_i_new=std_esa/std_somo*somo_i+mean_esa-mean_somo*std_esa/std_somo # method from ref.68
        esa_i_new_np[i,:]=esa_i_new
        # print(esa_i_new_np[i,:])
    era_esa_final=np.nanmean(esa_i_new_np,axis=0)
    return era_esa_final

In [3]:
import xarray as xr

# Load soil moisture datasets from NetCDF files
somo = xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_somo.nc')
gleam = xr.open_dataset('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_gleam.nc')
esa = xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_eascci.nc")
era5 = xr.open_dataset("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/swvl1.nc")

# Load the compiled multi-dataset soil moisture data
idata = np.load('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/sm_4dataset.npy')

# Load indices of samples containing NaN values and remove them from the data
nan_idx = np.load('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/conv3d_res/nan_idx.npy')
idata = np.delete(idata, nan_idx, axis=0)

# Load and clean the sample cell indices array
sa_x_cell = np.load("/User/homes/fhuang/lac_cnn/sa_x_cell.npy")
sa_x_cell = np.delete(sa_x_cell, nan_idx, axis=0)

# Initialize array for storing processed ESA soil moisture data
esa_sa = np.full((737, 216, 1), np.nan)

# Counter for tracking completely missing data cases
k = 0

# Process each of the 737 samples
for i in range(737):
    # Count number of NaN values in the ESA data for this sample
    nan_counts = np.isnan(idata[i, :, 6]).sum()
    
    # Case 1: All 216 time steps have missing ESA data
    if nan_counts == 216:
        print(i, k)  # Print sample index and counter
        k = k + 1
        
        # Use surrounding pixel mean to fill missing data
        x_cell = sa_x_cell[i]
        lat, lon = get_lat_lon(x_cell)
        esa_sa[i, :, 0] = surrounding_pixel_mean(somo, esa, lat, lon)
        
    # Case 2: Partial missing data (some time steps have valid data)
    elif (nan_counts < 216) & (nan_counts > 0):
        for j in range(216):
            esa_i = idata[i, j, 6]  # ESA data point
            somo_i = idata[i, j, 4]  # SOMo data point (used for imputation)
            
            if np.isnan(esa_i):
                # Calculate statistics for imputation
                std_esa = np.nanstd(idata[i, :, 6])  # Standard deviation of ESA data
                std_somo = np.nanstd(idata[i, :, 7])  # Standard deviation of SOMo data
                mean_esa = np.nanmean(idata[i, :, 6])  # Mean of ESA data
                mean_somo = np.nanmean(idata[i, :, 7])  # Mean of SOMo data
                
                # Linear regression imputation using SOMo data
                esa_i_new = std_esa / std_somo * somo_i + mean_esa - mean_somo * std_esa / std_somo
                esa_sa[i, j, :] = esa_i_new
            else:
                # Use existing valid ESA data
                esa_sa[i, j, :] = esa_i
        
    # Case 3: No missing data - use original values directly
    elif nan_counts == 0:
        esa_sa[i, :, 0] = idata[i, :, 6]

# Final cleanup: Replace any remaining NaN values with corresponding idata values
nan_coords = np.argwhere(np.isnan(esa_sa))
for coord in nan_coords:
    esa_sa[tuple(coord)] = idata[tuple(coord)]

# Save the processed ESA soil moisture data to file
np.save('/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/esa_sa.npy', esa_sa)

0 0


/Net/Groups/BGI/scratch/fhuang/.conda/envs/hfng/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_269277/3029193908.py:43: RuntimeWarning: Mean of empty slice
  mean_esa=np.nanmean(esa_i)
/tmp/ipykernel_269277/3029193908.py:44: RuntimeWarning: Mean of empty slice
  mean_somo=np.nanmean(somo_i)


36 1
44 2
73 3
74 4
78 5
79 6
81 7
89 8
90 9
91 10
92 11
93 12
97 13
98 14
99 15
100 16
104 17
105 18
106 19
107 20
108 21
110 22
111 23
112 24
116 25
117 26
118 27
119 28
120 29
124 30
125 31


/tmp/ipykernel_269277/3029193908.py:48: RuntimeWarning: Mean of empty slice
  era_esa_final=np.nanmean(esa_i_new_np,axis=0)


126 32
127 33
128 34
129 35
130 36
139 37
147 38
148 39
149 40
150 41
151 42
152 43
171 44
172 45
173 46
192 47
199 48
203 49
204 50
205 51
206 52
207 53
208 54
215 55
217 56
230 57
233 58
234 59
235 60
236 61
237 62
238 63
241 64
242 65
244 66
245 67
246 68
263 69
264 70
265 71
266 72
267 73
268 74
271 75
272 76
273 77
274 78
275 79
276 80
277 81
292 82
293 83
294 84
295 85
296 86
297 87
299 88
300 89
301 90
302 91
303 92
304 93
305 94
321 95
322 96
323 97
324 98
327 99
328 100
329 101
330 102
331 103
350 104
397 105
403 106
422 107
451 108
452 109
470 110
471 111
472 112
473 113
736 114
